# Example 2.2 / 2: Several Parameters in One List

Companion notebook for **2.2 Lists, Tables and Slices in Data Files** in the AMPL lab Documentation, section "Several Parameters in One List". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Several Parameters in One List

When parameters share the same index set, put their names after `param:` and then give one key
plus one value per header on each logical row:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                  # Products
param rate {PROD} > 0;     # Production rate in tons per hour
param profit {PROD};       # Profit per ton
param market {PROD} >= 0;  # Maximum sales in tons

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
rows = {
    "bands": (200, 25, 6000),
    "coils": (140, 30, 4000),
}
ampl.set["PROD"] = rows.keys()
ampl.param["rate"] = {p: values[0] for p, values in rows.items()}
ampl.param["profit"] = {p: values[1] for p, values in rows.items()}
ampl.param["market"] = {p: values[2] for p, values in rows.items()}

In [5]:
# run.py
print(ampl.get_data("rate", "profit", "market").to_pandas())

       rate  profit  market
bands   200      25    6000
coils   140      30    4000


### Pure AMPL: data.dat and model.run

The compact data-file form can define `PROD` and all three parameters together:

In [6]:
%%writefile data.dat
param: PROD: rate profit market :=
  bands 200 25 6000
  coils 140 30 4000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display rate, profit, market;

:      rate profit market    :=
bands   200    25    6000
coils   140    30    4000
;

